# **PCA Replication and Fashion-MNIST Application**

**Author:** Karla Hernandez Guerrero (4584600).    
**Course:** Data Wrangling (INFOMDW).        
**Assignment 2:** Paper Summarization--Data Preparation Report.            

## **Introduction**

Principal Component Analysis (PCA) summarizes correlated variables through a smaller set of uncorrelated components that retain as much variance as possible. This Jupyter notebook reproduces the World Happiness PCA example presented by Greenacre et al. (2022) and extends the workflow to Fashion-MNIST to explore dimensionality reduction in fashion-image data.

The original R workflow was adapted to Python using scikit-learn with generative AI assistance for code development and debugging. Validation focuses on comparing the reproduced World Happiness explained-variance percentages with the published values. Fashion-MNIST serves as an exploratory extension rather than a reproduction of an experiment from the paper.

## Objectives
1. Reproduce the reported World Happiness explained-variance percentages.
2. Apply standardized PCA to a reproducible sample of fashion images.
3. Report retained variance and visualize the first two components.
4. Share final results in GitHub repository.

  

## Setup

In [ ]:
# ============================================================
# 1. SETUP AND REPRODUCIBILITY
# ============================================================

# Import standard Python libraries
import sys
import json
import hashlib
import shutil
from pathlib import Path
from urllib.request import Request, urlopen
import time

# Import data-analysis and visualization libraries
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
import sklearn

# Import PCA, preprocessing, dataset, and notebook tools
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.datasets import fetch_openml
from IPython.display import display
try:
    from google.colab import files
except ImportError:
    files = None

# Create the folder used to save figures and tables
OUTPUT_DIR = Path("outputs") # organizes saved files
OUTPUT_DIR.mkdir(exist_ok=True)

# Define the Fashion-MNIST analysis settings
RANDOM_SEED = 42 # controls which fashion images are sampled
FASHION_SAMPLE_SIZE = 2000 # specifies how many images are analyzed
FASHION_COMPONENTS = 50 # specifies how many pc's are retained

# Display software versions for reproducibility
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

software_versions = {"Python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__, "scikit-learn": sklearn.__version__, "matplotlib": matplotlib.__version__}
(OUTPUT_DIR / "software_versions.json").write_text(json.dumps(software_versions, indent=2), encoding="utf-8")


## 1. World Happiness Reproduction

The **World Happiness Report 2021** dataset and original R code are available in the authors’ repository ([michaelgreenacre/PCA](https://github.com/michaelgreenacre/PCA)).

This analysis selects the five active variables used in the paper: social support, healthy life expectancy, freedom to make life choices, generosity, and perceptions of corruption. Other numeric columns are excluded because their inclusion would change the analysis.

The selected data are checked for missing values and standardized before PCA. The main validation compares the reproduced variance percentages with the reported values of 47.0% for PC1 and 24.5% for PC2.

Country scores and component coefficients are also displayed. Agreement in explained variance alone does not establish that every numerical or graphical output from the original analysis has been reproduced.

### How to load and check World Happiness data?

In [ ]:
# ============================================================
# 2. WORLD HAPPINESS: LOAD AND CHECK THE DATA
# ============================================================

# Define the local dataset path and original source
DATA_PATH = Path("world-happiness-report-2021.csv")

DATA_URL = (
    "https://raw.githubusercontent.com/michaelgreenacre/"
    "PCA/main/world-happiness-report-2021.csv"
)

# Download the CSV if it is not already in the working folder
if not DATA_PATH.exists():
    last_error = None
    for attempt in range(3):
        try:
            request = Request(DATA_URL, headers={"User-Agent": "PCA-course-notebook"})
            with urlopen(request, timeout=60) as response:
                payload = response.read()
            import io
            candidate = pd.read_csv(io.BytesIO(payload))
            if "Country name" not in candidate.columns:
                raise ValueError("Downloaded file is not the expected World Happiness CSV.")
            DATA_PATH.write_bytes(payload)
            break
        except Exception as exc:
            last_error = exc
            if attempt < 2:
                time.sleep(3)
    else:
        raise RuntimeError(
            "World Happiness download failed. Check internet access or upload "
            "world-happiness-report-2021.csv from the authors' repository and rerun."
        ) from last_error

# Load the dataset
df = pd.read_csv(DATA_PATH)

# Select the five active variables used in the paper
happiness_columns = [
    "Social support",
    "Healthy life expectancy",
    "Freedom to make life choices",
    "Generosity",
    "Perceptions of corruption"
]

required_columns = ["Country name"] + happiness_columns
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")
X_happiness = df[happiness_columns].apply(pd.to_numeric, errors="raise").copy()
assert (X_happiness.std() > 0).all(), "A selected indicator is constant."

# Check dataset dimensions and invalid values
assert X_happiness.shape == (149, 5), (
    f"Expected (149, 5), found {X_happiness.shape}."
)

assert np.isfinite(X_happiness.to_numpy()).all(), (
    "Missing or non-finite values detected."
)

#  Display the data checks
print("Countries:", X_happiness.shape[0])
print("Selected variables:", X_happiness.shape[1])

display(
    X_happiness.isna().sum().to_frame("Missing values")
)

# Calculate a checksum identifying the exact input file
dataset_hash = hashlib.sha256(
    DATA_PATH.read_bytes()
).hexdigest()

# Save the dataset source information
with open(OUTPUT_DIR / "happiness_data_source.json", "w") as f:
    json.dump({
        "source_url": DATA_URL,
        "filename": DATA_PATH.name,
        "sha256": dataset_hash,
        "variables": happiness_columns
    }, f, indent=2)

# Preserve the input CSV in the exported results
shutil.copy2(DATA_PATH, OUTPUT_DIR / DATA_PATH.name)

### How to fit PCA and compare with the paper?

In [ ]:
# ============================================================
# 3. WORLD HAPPINESS: FIT PCA AND COMPARE RESULTS
# ============================================================

# Center and scale the selected variables
happiness_scaler = StandardScaler(
    with_mean=True,
    with_std=True
)

X_happiness_scaled = happiness_scaler.fit_transform(
    X_happiness
)

# Extract all five components using full SVD
pca_happiness = PCA(
    n_components=None,
    svd_solver="full",
    whiten=False
)

happiness_scores = pca_happiness.fit_transform(
    X_happiness_scaled
)

# Calculate explained and cumulative variance
happiness_explained = pca_happiness.explained_variance_ratio_
happiness_cumulative = np.cumsum(happiness_explained)

# Create a variance summary for all five components
happiness_variance_table = pd.DataFrame({
    "Component": [f"PC{i}" for i in range(1, 6)],
    "Explained variance (%)": happiness_explained * 100,
    "Cumulative variance (%)": happiness_cumulative * 100
})

# Compare Python results with the paper's rounded values
comparison_table = pd.DataFrame({
    "Component": ["PC1", "PC2"],
    "Python (%)": happiness_explained[:2] * 100,
    "Paper (%)": [47.0, 24.5]
})

comparison_table["Difference (percentage points)"] = (
    comparison_table["Python (%)"]
    - comparison_table["Paper (%)"]
)

comparison_table["Matches at one decimal"] = (
    comparison_table["Python (%)"].round(1)
    == comparison_table["Paper (%)"]
)

# Display the numerical results
display(happiness_variance_table.round(3))
display(comparison_table.round(3))

# Find the number of components needed for each threshold
happiness_thresholds = {}

for threshold in [0.80, 0.90]:
    n_required = int(
        np.searchsorted(
            happiness_cumulative, threshold
        ) + 1
    )

    happiness_thresholds[threshold] = n_required

    print(
        f"Components required for at least "
        f"{threshold:.0%} variance: {n_required}"
    )

# Export the variance and comparison tables
happiness_variance_table.to_csv(
    OUTPUT_DIR / "happiness_variance.csv",
    index=False
)

comparison_table.to_csv(
    OUTPUT_DIR / "happiness_comparison.csv",
    index=False
)

### World Happiness: variance plots, country scores, and component coefficients


In [ ]:
# ============================================================
# 4. WORLD HAPPINESS: VISUALIZE AND EXPORT RESULTS
# ============================================================

# Define the component numbers
components = np.arange(1, 6)

# Plot individual and cumulative explained variance
fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(
    components,
    happiness_explained,
    marker="o",
    label="Individual"
)

ax.plot(
    components,
    happiness_cumulative,
    marker="o",
    label="Cumulative"
)

# Add the variance-retention reference line
ax.axhline(
    0.80,
    color="red",
    linestyle="--",
    label="80% threshold"
)

# Format and save the explained-variance plot
ax.set(
    xlabel="Principal component",
    ylabel="Explained variance ratio",
    title="PCA: World Happiness Data",
    xticks=components,
    ylim=(0, 1.05)
)

ax.legend()
fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "explained_variance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# Plot countries using their first two component scores
fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(
    happiness_scores[:, 0],
    happiness_scores[:, 1],
    alpha=0.7
)

# Label the axes with explained-variance percentages
ax.set_xlabel(
    f"PC1 ({happiness_explained[0]:.1%} explained variance)"
)

ax.set_ylabel(
    f"PC2 ({happiness_explained[1]:.1%} explained variance)"
)

ax.set_title("World Happiness: PCA Scores")
ax.set_aspect("equal", adjustable="datalim")
ax.grid(alpha=0.2)

# Save and display the score plot
fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "pca_score_plot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# Extract the component coefficients
happiness_coefficients = pd.DataFrame(
    pca_happiness.components_.T,
    index=happiness_columns,
    columns=[f"PC{i}" for i in range(1, 6)]
)

print("Component coefficients for PC1 and PC2:")
display(happiness_coefficients.iloc[:, :2].round(4))

# Save all component coefficients
happiness_coefficients.to_csv(
    OUTPUT_DIR / "happiness_component_coefficients.csv"
)

# Create a score table with country identifiers
happiness_score_table = pd.DataFrame(
    happiness_scores,
    columns=[f"PC{i}" for i in range(1, 6)]
)

happiness_score_table.insert(
    0,
    "Country name",
    df["Country name"].to_numpy()
)

# Save the country scores
happiness_score_table.to_csv(
    OUTPUT_DIR / "happiness_scores.csv",
    index=False
)

### What is the interpretation for the World Happiness PCA workflow?

The Python analysis reproduces the reported explained-variance percentages at one-decimal precision: 47.0% for PC1 and 24.5% for PC2. Together, these components retain approximately 71.5% of the standardized variance. Three components exceed the 80% cumulative variance, while four exceed 90%.

The score plot represents countries using their first two component scores. It provides a compact view of the observations but leaves approximately 28.5% of the variance outside the displayed space.   

The comparison supports agreement in the reported variance percentages. It does not establish that every country score, coefficient, or plotting convention has been independently matched to the original R analysis.

## 2. Fashion-MNIST Application

**Fashion-MNIST** contains 70,000 grayscale images across 10 clothing and accessory categories. Each 28 x 28 image is represented by 784 pixel features. The dataset is retrieved using scikit-learn `fetch_openml()` function.

A sample of 2,000 images is selected without replacement using the seed 42. The sample is drawn from the complete fetched dataset because this analysis is descriptive and does not evaluate a predictive model.

Pixel features are standardized to maintain a consistent preprocessing workflow with the World Happiness analysis. This is a methodological choice rather than a requirement for all image PCA applications. Standardization changes the relative weighting of pixel features.

Fifty components are retained as a predefined exploratory setting, not an optimized choice. Their cumulative explained variance is reported, and the first two components are visualized. Clothing labels are used only to color the plot, not to fit PCA.

The 80% and 90% thresholds are checked within the fitted 50 components. If a threshold is not reached, the notebook does not claim how many additional components are needed. Image reconstructions are not included in this notebook.


### How do we prepare the Fashion-MNIST data for PCA?

In [ ]:
# ============================================================
# 5. FASHION-MNIST: LOAD, SAMPLE, STANDARDIZE, AND FIT PCA
# ============================================================

# Fetch the specified dataset version from OpenML
mnist = fetch_openml(
    "Fashion-MNIST",
    version=1,
    as_frame=False,
    parser="auto",
    n_retries=3,
    delay=3
)

# Convert pixel features and labels to numeric arrays
X_fashion_all = mnist.data.astype(np.float32)
y_fashion_all = mnist.target.astype(int)

# Check the dimensions of the fetched dataset
assert X_fashion_all.shape == (70000, 784), (
    f"Unexpected dataset shape: {X_fashion_all.shape}"
)

assert y_fashion_all.shape == (70000,)
assert set(np.unique(y_fashion_all)) == set(range(10))
assert 2 <= FASHION_SAMPLE_SIZE <= len(X_fashion_all)
assert 2 <= FASHION_COMPONENTS <= min(FASHION_SAMPLE_SIZE, 784)

# Select a reproducible sample without replacement
rng = np.random.default_rng(RANDOM_SEED)

fashion_sample_indices = rng.choice(
    len(X_fashion_all),
    size=FASHION_SAMPLE_SIZE,
    replace=False
)

X_fashion = X_fashion_all[fashion_sample_indices]
y_fashion = y_fashion_all[fashion_sample_indices]

# Check the sampled data
assert X_fashion.shape == (FASHION_SAMPLE_SIZE, 784)
assert np.isfinite(X_fashion).all()

print("Sampled images:", X_fashion.shape[0])
print("Pixel features:", X_fashion.shape[1])

# Save the selected row indices
np.save(
    OUTPUT_DIR / "fashion_sample_indices.npy",
    fashion_sample_indices
)

# Record the fetched dataset details
with open(OUTPUT_DIR / "fashion_data_source.json", "w") as f:
    json.dump({
        "dataset_name": "Fashion-MNIST",
        "version": 1,
        "openml_data_id": mnist.details.get("id"),
        "sample_size": FASHION_SAMPLE_SIZE,
        "sampling_seed": RANDOM_SEED,
        "sampling": "Without replacement"
    }, f, indent=2)

# Center and scale the pixel features
fashion_scaler = StandardScaler(
    with_mean=True,
    with_std=True
)

X_fashion_scaled = fashion_scaler.fit_transform(
    X_fashion
)

# Fit PCA using 50 components and full SVD
pca_fashion = PCA(
    n_components=FASHION_COMPONENTS,
    svd_solver="full",
    whiten=False
)

fashion_scores = pca_fashion.fit_transform(
    X_fashion_scaled
)

# Calculate individual and cumulative explained variance
fashion_explained = pca_fashion.explained_variance_ratio_
fashion_cumulative = np.cumsum(fashion_explained)

# Summarize variance retained
fashion_summary = pd.DataFrame({
    "Representation": [
        "PC1",
        "PC2",
        "First two combined",
        "All 50 combined"
    ],
    "Explained variance (%)": [
        fashion_explained[0] * 100,
        fashion_explained[1] * 100,
        fashion_cumulative[1] * 100,
        fashion_cumulative[-1] * 100
    ]
})

display(fashion_summary.round(3))

# Export the numerical summary
fashion_summary.to_csv(
    OUTPUT_DIR / "fashion_variance_summary.csv",
    index=False
)

# Thresholds are evaluated only within the fitted 50-component model.
fashion_threshold_rows = []
for threshold in (0.80, 0.90):
    reached = np.flatnonzero(fashion_cumulative >= threshold)
    count = int(reached[0] + 1) if reached.size else None
    fashion_threshold_rows.append({
        "Target variance (%)": threshold * 100,
        "Components required within fitted model": count,
        "Status": "Reached" if count is not None else f"Not reached within {FASHION_COMPONENTS} components"
    })
fashion_threshold_table = pd.DataFrame(fashion_threshold_rows)
display(fashion_threshold_table)
fashion_threshold_table.to_csv(OUTPUT_DIR / "fashion_variance_thresholds.csv", index=False)


### How do we visualize PCA using fashion data?

In [ ]:
# ============================================================
# 6. FASHION-MNIST: VISUALIZE THE FIRST TWO COMPONENTS
# ============================================================

# Define category names in numerical label order
category_names = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]

# Display the number of sampled images in each category
category_counts = pd.Series(
    y_fashion
).value_counts().sort_index()

category_count_table = pd.DataFrame({
    "Category": [
        category_names[i] for i in category_counts.index
    ],
    "Sample count": category_counts.to_numpy()
})

display(category_count_table)

category_count_table.to_csv(
    OUTPUT_DIR / "fashion_category_counts.csv",
    index=False
)

# Set the categorical color palette
colors = plt.get_cmap("tab10")

# Create the plotting area
fig, ax = plt.subplots(figsize=(9, 6))

# Plot each category using its PC1 and PC2 scores
# Category labels were not used to fit PCA.
for category_id, category_name in enumerate(category_names):
    mask = y_fashion == category_id

    ax.scatter(
        fashion_scores[mask, 0],
        fashion_scores[mask, 1],
        color=colors(category_id),
        label=category_name,
        s=12,
        alpha=0.65
    )

# Add explained-variance percentages to the axes
ax.set_xlabel(
    f"PC1 ({fashion_explained[0]:.1%} explained variance)"
)

ax.set_ylabel(
    f"PC2 ({fashion_explained[1]:.1%} explained variance)"
)

# Set the title and equal axis units
ax.set_title("Fashion-MNIST: First Two Principal Components")
ax.set_aspect("equal", adjustable="datalim")

# Add category names in an external legend
ax.legend(
    title="Clothing category",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

# Save the figure before displaying it
fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "fashion_pca_scatterplot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# Print the variance retained by the full reduced representation
print(
    f"The {FASHION_COMPONENTS}-component representation retains "
    f"{fashion_cumulative[-1]:.2%} of standardized variance."
)

# Print the variance represented in the scatterplot
print(
    f"The first two plotted components retain "
    f"{fashion_cumulative[1]:.2%} of standardized variance."
)


### Interpreting the Fashion-MNIST variance results


In [ ]:
# ============================================================
# 7. FASHION-MNIST: GENERATE A NUMERICAL INTERPRETATION
# ============================================================

# Insert the computed variance percentages into a short summary
fashion_interpretation = (
    f"PCA reduced the 784 pixel features to "
    f"{FASHION_COMPONENTS} components, retaining "
    f"{fashion_cumulative[-1] * 100:.2f}% of the standardized "
    f"variance. The first two components retained "
    f"{fashion_cumulative[1] * 100:.2f}%. The scatterplot displays "
    f"only these two components, rather than the complete "
    f"{FASHION_COMPONENTS}-component representation. "
    f"Category labels were used for visualization only; "
    f"this analysis does not evaluate classification accuracy."
)

print(fashion_interpretation)

# Save the interpretation alongside the numerical outputs
(OUTPUT_DIR / "fashion_interpretation.txt").write_text(
    fashion_interpretation,
    encoding="utf-8"
)

## **Discussion**

As it was previously mentioned, the World Happiness analysis reproduces the paper's reported PC1 and PC2 explained-variance percentages at one-decimal precision. Selecting the correct five active variables was essential: including every numeric column in the initial analysis produced a different PCA. The final comparison validates the reported variance percentages rather than every numerical or graphical output from the original analysis.

Fashion-MNIST provides a separate application of the workflow. Its variance percentages are not expected to match those of World Happiness because the datasets have different variables and structures. The category-colored plot provides an exploratory view of the fashion images, not a measure of classification performance.

The fashion application is limited to one sample, one scaling approach, and a predefined setting of 50 components. No claim is made that this setting is optimal. The notebook does not measure changes in model accuracy or computational efficiency.

A future predictive evaluation would require fitting preprocessing on training data and evaluating a downstream model on held-out observations. That task is outside the scope of this reproduction.

## **Parameter settings**

This section lists all parameters required to run the code and specifies the values used in the PCA application to the World Happiness and Fashion-MNIST data.

In [ ]:
# ============================================================
# 8. RECORD THE PARAMETER SETTINGS
# ============================================================

# Create a side-by-side record of the analysis settings
parameter_table = pd.DataFrame({
    "Setting": [
        "Dataset",
        "Observations",
        "Input features",
        "Sampling",
        "Sampling seed",
        "Centering",
        "Scaling",
        "PCA components",
        "SVD solver",
        "Whitening",
        "Displayed components",
        "Variance thresholds"
    ],
    "World Happiness": [
        "Authors' World Happiness Report 2021 CSV",
        "149",
        "Five selected indicators",
        "All rows",
        "Not applicable",
        "Yes",
        "Unit variance",
        "All five",
        "full",
        "False",
        "PC1 and PC2",
        "80% and 90%"
    ],
    "Fashion-MNIST": [
        "OpenML Fashion-MNIST, version 1",
        str(FASHION_SAMPLE_SIZE),
        "784 pixel features",
        "Without replacement from all 70,000 images",
        str(RANDOM_SEED),
        "Yes",
        "Unit variance",
        str(FASHION_COMPONENTS),
        "full",
        "False",
        "PC1 and PC2",
        "80% and 90%; reported only if reached within 50 components"
    ]
})

# Display the settings
display(parameter_table)

# Export the settings
parameter_table.to_csv(
    OUTPUT_DIR / "parameter_settings.csv",
    index=False
)

##  **Reproducibility and code availability**

For a better understanding of how to run the code, all cells must be run from top to bottom in a fresh Google colab runtime. Internet access is required to retrieve the datasets.

The World Happiness CSV is downloaded if a local copy is absent. The exact input file is preserved in the exported results, and its SHA-256 checksum is recorded.

The Fashion-MNIST sample used seed 42, and its selected row indices are saved. Both analyses specify the full SVD solver and disable whitening. Software versions, parameter settings, figures, and numerical tables are exported with the results.

The notebook uses scikit-learn's existing PCA implementation. It is a Python reproduction of the analysis workflow, not a from-scratch implementation of the PCA algorithm.

GitHub repository: [Insert your repository URL before submission.]

## **Output Download**

In [ ]:
# ============================================================
# 9. EXPORT AND DOWNLOAD THE RESULTS
# ============================================================

# List the generated output files
print("Files included in the results archive:")

for output_file in sorted(OUTPUT_DIR.iterdir()):
    print(f"- {output_file.name}")

# Package all outputs into a ZIP archive
archive_path = shutil.make_archive(
    "pca_results",
    "zip",
    root_dir=OUTPUT_DIR
)

# Download the archive to your computer
if files is not None:
    files.download(archive_path)
else:
    print(f"Results archive saved locally: {archive_path}")

## **References**  
Greenacre, M., Groenen, P. J. F., Hastie, T., d’Enza, A. I., Markos, A., & Tuzhilina, E. (2022). Principal component analysis. *Nature Reviews Methods Primers*, 2, 100.

Xiao, H., Rasul, K., & Vollgraf, R. (2017). Fashion-MNIST:
A novel image dataset for benchmarking machine learning algorithms.
arXiv:1708.07747.
https://arxiv.org/abs/1708.07747

### Data and software resources

Greenacre et al. dataset and R scripts:
https://github.com/michaelgreenacre/PCA

Fashion-MNIST dataset:
https://github.com/zalandoresearch/fashion-mnist

scikit-learn PCA documentation:
https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html

scikit-learn StandardScaler documentation:
https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html